# Softmax Regression from Scratch

**Interview-focused reference notebook** -- Numerically stable softmax, multi-class classification, temperature scaling.

## Core Intuition

Softmax regression generalizes logistic regression to $K$ classes. It is equivalent to a neural network with zero hidden layers.

**Model:** For input $\mathbf{x} \in \mathbb{R}^d$, compute logits $\mathbf{z} = W\mathbf{x} + \mathbf{b}$ where $W \in \mathbb{R}^{K \times d}$, then:

$$P(y=k|\mathbf{x}) = \text{softmax}(\mathbf{z})_k = \frac{e^{z_k}}{\sum_{j=1}^K e^{z_j}}$$

**Key properties:**
- Output sums to 1 (valid probability distribution)
- Invariant to adding a constant: $\text{softmax}(\mathbf{z}) = \text{softmax}(\mathbf{z} + c)$
- Reduces to sigmoid for $K=2$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

## 1. Softmax Function -- Naive vs Numerically Stable

In [ ]:
def softmax_naive(z):
    """Naive softmax -- will overflow for large z."""
    exp_z = np.exp(z)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

def softmax_stable(z):
    """Numerically stable softmax: subtract max before exp.
    
    Uses the invariance property: softmax(z) = softmax(z - c) for any c.
    Choosing c = max(z) prevents overflow.
    """
    z_shifted = z - np.max(z, axis=-1, keepdims=True)
    exp_z = np.exp(z_shifted)
    return exp_z / exp_z.sum(axis=-1, keepdims=True)

# Demonstrate the overflow problem
z_safe = np.array([1.0, 2.0, 3.0])
z_dangerous = np.array([1000.0, 1001.0, 1002.0])

print("Safe input [1, 2, 3]:")
print(f"  Naive:  {softmax_naive(z_safe)}")
print(f"  Stable: {softmax_stable(z_safe)}")

print("\nDangerous input [1000, 1001, 1002]:")
import warnings
with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    print(f"  Naive:  {softmax_naive(z_dangerous)}")  # NaN!
print(f"  Stable: {softmax_stable(z_dangerous)}")  # Works fine

## 2. Log-Sum-Exp Trick

When computing $\log \sum_k e^{z_k}$ (which appears in the loss), use:

$$\log \sum_k e^{z_k} = m + \log \sum_k e^{z_k - m}$$

where $m = \max_k z_k$. This prevents overflow in the $\exp$ and underflow in the $\log$.

In [ ]:
def logsumexp(z, axis=-1, keepdims=False):
    """Numerically stable log-sum-exp."""
    z_max = np.max(z, axis=axis, keepdims=True)
    result = z_max + np.log(np.sum(np.exp(z - z_max), axis=axis, keepdims=True))
    if not keepdims:
        result = result.squeeze(axis=axis)
    return result

def log_softmax(z):
    """Numerically stable log-softmax using log-sum-exp trick."""
    return z - logsumexp(z, axis=-1, keepdims=True)

# Verify
z = np.array([[1000.0, 1001.0, 1002.0],
              [-1000.0, -999.0, -998.0]])

print("Log-softmax (stable):")
print(log_softmax(z))
print("\nSoftmax from log-softmax:")
print(np.exp(log_softmax(z)))
print("\nDirect stable softmax:")
print(softmax_stable(z))
print("\nAll match!")

## 3. Synthetic Multi-Class Data

In [ ]:
from sklearn.datasets import make_blobs, make_classification

# 3-class problem
X_3, y_3 = make_blobs(n_samples=450, centers=3, n_features=2,
                       random_state=42, cluster_std=1.2)

# 5-class problem (harder)
X_5, y_5 = make_blobs(n_samples=500, centers=5, n_features=2,
                       random_state=42, cluster_std=1.5)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, X, y, title in [(axes[0], X_3, y_3, '3 Classes'), (axes[1], X_5, y_5, '5 Classes')]:
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis', alpha=0.7, edgecolors='k', s=30)
    ax.set_title(title)
    ax.legend(*scatter.legend_elements(), title='Class')
plt.tight_layout()
plt.show()

## 4. Full Softmax Regression Implementation

**Gradient derivation:**

$$L = -\frac{1}{n}\sum_{i=1}^{n} \log P(y_i | \mathbf{x}_i) = -\frac{1}{n}\sum_{i=1}^{n} \sum_{k=1}^{K} y_{ik} \log \hat{y}_{ik}$$

$$\nabla_{W_k} L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_{ik} - y_{ik}) \mathbf{x}_i$$

In matrix form: $\nabla_W L = \frac{1}{n} X^T (\hat{Y} - Y)$

where $\hat{Y} = \text{softmax}(XW)$ and $Y$ is one-hot encoded.

In [ ]:
class SoftmaxRegression:
    """Multi-class classification via softmax regression."""
    
    def __init__(self, lr=0.1, n_iters=1000, reg=0.0):
        self.lr = lr
        self.n_iters = n_iters
        self.reg = reg  # L2 regularization
        self.loss_history = []
    
    def _softmax(self, Z):
        Z_shifted = Z - Z.max(axis=1, keepdims=True)
        exp_Z = np.exp(Z_shifted)
        return exp_Z / exp_Z.sum(axis=1, keepdims=True)
    
    def _one_hot(self, y, K):
        n = y.shape[0]
        Y = np.zeros((n, K))
        Y[np.arange(n), y] = 1
        return Y
    
    def _cross_entropy_loss(self, Y, probs):
        """Numerically stable cross-entropy using log-softmax."""
        eps = 1e-12
        return -np.mean(np.sum(Y * np.log(probs + eps), axis=1))
    
    def fit(self, X, y):
        n, d = X.shape
        self.classes_ = np.unique(y)
        K = len(self.classes_)
        
        # Add bias column
        X_b = np.column_stack([np.ones(n), X])
        Y = self._one_hot(y, K)
        
        # Initialize weights: (d+1) x K
        self.W_ = np.zeros((d + 1, K))
        
        for i in range(self.n_iters):
            # Forward pass
            logits = X_b @ self.W_  # n x K
            probs = self._softmax(logits)  # n x K
            
            # Loss
            loss = self._cross_entropy_loss(Y, probs)
            if self.reg > 0:
                loss += 0.5 * self.reg * np.sum(self.W_[1:] ** 2)
            self.loss_history.append(loss)
            
            # Gradient: (1/n) X^T (P - Y)
            grad = (1.0 / n) * X_b.T @ (probs - Y)
            if self.reg > 0:
                grad[1:] += self.reg * self.W_[1:]  # don't regularize bias
            
            # Update
            self.W_ -= self.lr * grad
        
        return self
    
    def predict_proba(self, X):
        X_b = np.column_stack([np.ones(X.shape[0]), X])
        return self._softmax(X_b @ self.W_)
    
    def predict(self, X):
        return self.classes_[np.argmax(self.predict_proba(X), axis=1)]

# Train on 3-class data
model_3 = SoftmaxRegression(lr=0.1, n_iters=1000).fit(X_3, y_3)
acc_3 = np.mean(model_3.predict(X_3) == y_3)
print(f"3-class accuracy: {acc_3:.4f}")

# Train on 5-class data
model_5 = SoftmaxRegression(lr=0.05, n_iters=2000).fit(X_5, y_5)
acc_5 = np.mean(model_5.predict(X_5) == y_5)
print(f"5-class accuracy: {acc_5:.4f}")

## 5. Training Loss Convergence

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(model_3.loss_history, linewidth=2)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('Cross-Entropy Loss')
axes[0].set_title('3-Class Training Loss')
axes[0].grid(True, alpha=0.3)

axes[1].plot(model_5.loss_history, linewidth=2, color='orange')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Cross-Entropy Loss')
axes[1].set_title('5-Class Training Loss')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Decision Region Visualization

In [ ]:
def plot_decision_regions(model, X, y, ax, title='', resolution=0.1):
    """Plot multi-class decision regions."""
    x_min, x_max = X[:, 0].min() - 2, X[:, 0].max() + 2
    y_min, y_max = X[:, 1].min() - 2, X[:, 1].max() + 2
    xx, yy = np.meshgrid(np.arange(x_min, x_max, resolution),
                          np.arange(y_min, y_max, resolution))
    Z = model.predict(np.column_stack([xx.ravel(), yy.ravel()]))
    Z = Z.reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='viridis')
    scatter = ax.scatter(X[:, 0], X[:, 1], c=y, cmap='viridis',
                         edgecolors='k', s=30, alpha=0.7)
    ax.set_title(title)
    return scatter

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
plot_decision_regions(model_3, X_3, y_3, axes[0], '3-Class Decision Regions')
plot_decision_regions(model_5, X_5, y_5, axes[1], '5-Class Decision Regions')
plt.tight_layout()
plt.show()
print("All decision boundaries are linear (hyperplanes) -- softmax regression is a linear classifier.")

## 7. Probability Maps -- Confidence Visualization

In [ ]:
# Show prediction confidence (max probability) across the space
h = 0.1
x_min, x_max = X_3[:, 0].min() - 2, X_3[:, 0].max() + 2
y_min, y_max = X_3[:, 1].min() - 2, X_3[:, 1].max() + 2
xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
grid = np.column_stack([xx.ravel(), yy.ravel()])

probs = model_3.predict_proba(grid)
confidence = probs.max(axis=1).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(8, 6))
cs = ax.contourf(xx, yy, confidence, levels=20, cmap='RdYlGn', alpha=0.8)
plt.colorbar(cs, ax=ax, label='Max Probability (Confidence)')
ax.scatter(X_3[:, 0], X_3[:, 1], c=y_3, cmap='viridis', edgecolors='k', s=40, alpha=0.8)
ax.set_title('Prediction Confidence Map -- Low confidence near boundaries')
plt.tight_layout()
plt.show()

## 8. Softmax Temperature

**Temperature scaling:** $\text{softmax}(\mathbf{z}/T)$

- $T=1$: standard softmax
- $T \to 0$: approaches argmax (hard/one-hot distribution)
- $T \to \infty$: approaches uniform distribution (maximum entropy)

**Interview context:** Temperature is used in:
- Knowledge distillation (soft targets from teacher model)
- Language model sampling (control creativity vs. determinism)
- Calibration of neural network confidence

In [ ]:
logits = np.array([2.0, 1.0, 0.5])
temperatures = [0.1, 0.5, 1.0, 2.0, 5.0, 10.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart for different temperatures
x = np.arange(3)
width = 0.12
for i, T in enumerate(temperatures):
    probs = softmax_stable(logits / T)
    axes[0].bar(x + i * width, probs, width, label=f'T={T}', alpha=0.8)

axes[0].set_xticks(x + width * 2.5)
axes[0].set_xticklabels(['Class 0', 'Class 1', 'Class 2'])
axes[0].set_ylabel('Probability')
axes[0].set_title('Softmax Output at Different Temperatures')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3, axis='y')

# Entropy vs temperature
T_range = np.linspace(0.1, 10, 100)
entropies = []
for T in T_range:
    p = softmax_stable(logits / T)
    entropy = -np.sum(p * np.log(p + 1e-12))
    entropies.append(entropy)

axes[1].plot(T_range, entropies, 'b-', linewidth=2)
axes[1].axhline(y=np.log(3), color='r', linestyle='--', label=f'Max entropy = ln(3) = {np.log(3):.3f}')
axes[1].set_xlabel('Temperature T')
axes[1].set_ylabel('Entropy of softmax output')
axes[1].set_title('Entropy vs Temperature')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Low T -> peaked (confident/greedy). High T -> flat (uncertain/exploratory).")

## 9. Softmax vs Sigmoid for Multi-Class

In [ ]:
# Compare: K independent sigmoids (multi-label) vs softmax (multi-class)
logits_example = np.array([2.0, 1.0, 0.5])

# Sigmoid: each output independent
sigmoid_probs = 1 / (1 + np.exp(-logits_example))

# Softmax: outputs sum to 1
softmax_probs = softmax_stable(logits_example)

print("Logits:          ", logits_example)
print(f"Sigmoid outputs:  {sigmoid_probs.round(4)}  (sum = {sigmoid_probs.sum():.4f})")
print(f"Softmax outputs:  {softmax_probs.round(4)}  (sum = {softmax_probs.sum():.4f})")
print()
print("Sigmoid: each class independent. Sum != 1. Good for MULTI-LABEL (image can have cat AND dog).")
print("Softmax: classes compete. Sum = 1. Good for MULTI-CLASS (image is cat OR dog, not both).")

## 10. Softmax Regression = 0-Hidden-Layer Neural Net

In [ ]:
# Show that softmax regression is identical to a single-layer network
print("Architecture comparison:")
print()
print("Softmax Regression:")
print("  Input (d) -> Linear(d, K) -> Softmax -> Cross-Entropy Loss")
print()
print("1-Layer Neural Net:")
print("  Input (d) -> Linear(d, h) -> ReLU -> Linear(h, K) -> Softmax -> Cross-Entropy Loss")
print()
print("Softmax regression can only learn linear decision boundaries.")
print("Adding hidden layers enables nonlinear boundaries.")
print()
print("This is why softmax regression is the starting point for understanding neural nets.")

## 11. Sklearn Comparison

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Compare on both datasets
for name, X, y, model in [
    ('3-class', X_3, y_3, model_3),
    ('5-class', X_5, y_5, model_5)
]:
    sk_model = LogisticRegression(multi_class='multinomial', max_iter=1000).fit(X, y)
    our_acc = accuracy_score(y, model.predict(X))
    sk_acc = sk_model.score(X, y)
    print(f"{name}: Our accuracy = {our_acc:.4f}, sklearn accuracy = {sk_acc:.4f}")

## 12. Jacobian of Softmax

The Jacobian of the softmax function is important for backpropagation:

$$\frac{\partial \text{softmax}(z)_i}{\partial z_j} = \begin{cases} p_i(1 - p_i) & \text{if } i = j \\ -p_i p_j & \text{if } i \neq j \end{cases}$$

In matrix form: $J = \text{diag}(\mathbf{p}) - \mathbf{p}\mathbf{p}^T$

In [ ]:
def softmax_jacobian(z):
    """Compute the Jacobian matrix of the softmax function."""
    p = softmax_stable(z)
    return np.diag(p) - np.outer(p, p)

# Verify with numerical differentiation
z_test = np.array([1.0, 2.0, 3.0])
J_analytical = softmax_jacobian(z_test)

# Numerical Jacobian
eps = 1e-5
J_numerical = np.zeros((3, 3))
for j in range(3):
    z_plus = z_test.copy()
    z_plus[j] += eps
    z_minus = z_test.copy()
    z_minus[j] -= eps
    J_numerical[:, j] = (softmax_stable(z_plus) - softmax_stable(z_minus)) / (2 * eps)

print("Analytical Jacobian:")
print(J_analytical.round(6))
print("\nNumerical Jacobian:")
print(J_numerical.round(6))
print(f"\nMax absolute difference: {np.max(np.abs(J_analytical - J_numerical)):.2e}")

## Interview Quick Reference

| Question | Answer |
|----------|--------|
| Softmax vs sigmoid for multi-class? | Softmax: classes are mutually exclusive (sum to 1). Sigmoid: classes are independent (multi-label). For K=2, softmax reduces to sigmoid. |
| Softmax temperature -- what does it do? | $T<1$: sharpens distribution (more confident). $T>1$: flattens distribution (more uniform). $T \to 0$: argmax. $T \to \infty$: uniform. Used in distillation, LLM sampling, calibration. |
| How to make softmax numerically stable? | Subtract $\max(z)$ before exponentiating. Uses the translation invariance property. |
| What is log-sum-exp? | $\log \sum e^{z_k} = m + \log \sum e^{z_k - m}$ where $m = \max(z)$. Prevents overflow/underflow. |
| Softmax regression vs neural net? | Softmax regression = 0 hidden layer neural net. Can only learn linear boundaries. Adding layers enables nonlinear boundaries. |
| Gradient of CE + softmax? | Elegantly simplifies to $(\hat{y} - y)$ -- same form as linear/logistic regression. |